# Official scratch 3D EfficientNet-B0 — complete five-fold ImageID workflow
This single notebook reuses the validated Fold-1 checkpoint, trains Folds 2–5 sequentially, freezes all validation-selected checkpoints, and only then evaluates the fixed 719-ImageID test cohort. No tuning, augmentation, imbalance method, or pretrained weights are used.

In [1]:
import os
GPU2_UUID = 'GPU-db657392-ab6d-94b9-02fd-9db3a5e31c3d'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU2_UUID
os.environ['PYTHONHASHSEED'] = '42'

import gc, hashlib, json, random, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import monai, nbformat, numpy as np, pandas as pd, seaborn as sns, torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, RandomSampler

PROJECT_ROOT = Path('/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM')
BASELINE_NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks' / 'Baselines'
sys.path.insert(0, str(BASELINE_NOTEBOOK_DIR))
from run_Fold1_DenseNet_EfficientNet_Baselines import (
    Fold1MRIDataset, build_model, calculate_metrics, json_dump, read_split,
    run_phase, seed_all, seed_worker, sha256_file, state_checksum,
)

RESULTS_BASE = PROJECT_ROOT / 'Results' / 'Baselines'
RESULTS_ROOT = RESULTS_BASE / '3DEfficientNet_ImageID_5Fold'
SUMMARY_DIR = RESULTS_ROOT / 'summary'
FOLDS_ROOT = Path('/home/faizaan/AD/projects/HandlingImbalance/folds')
FOLD1_SOURCE = RESULTS_BASE / '3DEfficientNet_ImageID_Fold1'
CONFIG_FREEZE_PATH = RESULTS_BASE / 'EfficientNet_B0_5Fold_Config_Freeze.json'
HELPER_PATH = BASELINE_NOTEBOOK_DIR / 'run_Fold1_DenseNet_EfficientNet_Baselines.py'
TECHNICAL_RESUME = RESULTS_ROOT.exists()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
NUM_CLASSES = 6
LABEL_MAP = {'AD': 0, 'CN': 1, 'EMCI': 2, 'LMCI': 3, 'MCI': 4, 'SMC': 5}
CLASS_NAMES = ['AD', 'CN', 'EMCI', 'LMCI', 'MCI', 'SMC']
PROBABILITY_COLUMNS = [f'probability {name}' for name in CLASS_NAMES]
BATCH_SIZE = 4
NUM_WORKERS = 2
PREFETCH_FACTOR = 1
MAX_EPOCHS = 40
EARLY_STOP_PATIENCE = 10
MIN_DELTA = 1e-4
INITIAL_LR = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
PLOT_DPI = 600
PRETRAINED = False
FOLDS = [1, 2, 3, 4, 5]
EXPECTED_PARAMETERS = 4_696_066
EXPECTED_TEST_SHA256 = 'c58d88969c828ac806a6559798841effce482fd6081b5fa56bce5c91c66fc268'
SPLIT_HASHES = {
  1: {'train': '23c2c029b673293bbd313c79eaf220efdd794bf7843118a664f1f8c5bbb038c9', 'val': '0b964df6ec6747aa62e6ce2b631c4e439697512f96ba6d1c824bb6e0eadfa5a6'},
  2: {'train': '6c97819dbdc369a8c30874fae93348692557f7f6f86cf93afa0579b719a439dd', 'val': '907bb3d51aaa666d5597ed94d62fc5202b7dd8abc3dc0ba5c54029ad6f31395a'},
  3: {'train': 'eb3f3626780cdfa8776c6623decadbb2f5e51eb6a073984c29dba81a2143c829', 'val': '9ac0f99685a6febbdc59fc7d6668dec7f37ad01e25c8de9a5ff68c1a95fbfc12'},
  4: {'train': '711547f17fa1bf34fd180a9080b1533b311d3a801106b979875c4d9cfd10f652', 'val': '11bf0b60f5d3f55a1b47ef635cf1a6b6ab8b59ec9998762fd550be455e612cb5'},
  5: {'train': 'f67f7575ed2d485c1fd8852105edb34a66acb96bf29edb24bb99da541b38e393', 'val': 'a8b12cadd249e30916208c98f4f0f6da93d586787ce589988f39671e5c8b5ba0'},
}
EXPECTED_ROWS = {1: (2298,575), 2: (2298,575), 3: (2298,575), 4: (2299,574), 5: (2299,574)}

seed_all(SEED)
parameter_probe = build_model('E')
PARAMETERS = sum(p.numel() for p in parameter_probe.parameters())
INITIALIZATION_SHA256 = state_checksum(parameter_probe)
del parameter_probe
assert PARAMETERS == EXPECTED_PARAMETERS
assert monai.__version__ == '1.5.1'
assert not PRETRAINED and torch.cuda.is_available() and torch.cuda.device_count() == 1
assert torch.cuda.is_bf16_supported()

test_id_sets = []
for fold in FOLDS:
    test_csv = FOLDS_ROOT / f'fold_{fold}' / 'test.csv'
    assert sha256_file(test_csv) == EXPECTED_TEST_SHA256
    ids = pd.read_csv(test_csv, usecols=['ImageID'], dtype={'ImageID': str}).ImageID
    assert len(ids) == 719 and ids.nunique() == 719
    test_id_sets.append(set(ids))
assert all(ids == test_id_sets[0] for ids in test_id_sets[1:])

CONFIG = {
 'EFFICIENTNET_CONFIG_FROZEN': True, 'frozen_utc': datetime.now(timezone.utc).isoformat(),
 'monai_version': monai.__version__, 'architecture': 'Official MONAI EfficientNetBN efficientnet-b0, spatial_dims=3, in_channels=1, num_classes=6',
 'parameters': PARAMETERS, 'pretrained': False, 'seed': SEED, 'label_mapping': LABEL_MAP,
 'optimizer': 'AdamW', 'initial_lr': INITIAL_LR, 'weight_decay': WEIGHT_DECAY,
 'scheduler': 'ReduceLROnPlateau(mode=max, validation Macro F1, factor=1/3, patience=3)',
 'batch_size': BATCH_SIZE, 'effective_batch_size': BATCH_SIZE, 'precision': 'BF16',
 'loss': 'plain CrossEntropyLoss; no class weights/focal/label smoothing',
 'sampler': 'natural shuffled RandomSampler; no weighting', 'augmentation': False,
 'preprocessing': 'existing bias-corrected MRI; nan_to_num; per-volume min-max; [1,128,128,128]',
 'max_epochs': MAX_EPOCHS, 'early_stop_patience': EARLY_STOP_PATIENCE, 'min_delta': MIN_DELTA,
 'gradient_clip_norm': GRAD_CLIP_NORM, 'checkpoint_criterion': 'validation Macro F1 only',
 'fixed_test_membership_sha256': EXPECTED_TEST_SHA256, 'fixed_test_unique_imageids': 719,
 'fold_split_hashes': SPLIT_HASHES, 'validated_fold1_checkpoint_sha256': sha256_file(FOLD1_SOURCE/'best_model.pt'),
 'validated_implementation_source': str(HELPER_PATH), 'validated_implementation_sha256': sha256_file(HELPER_PATH),
 'initialization_sha256': INITIALIZATION_SHA256, 'fold1_reused': True, 'folds_2_to_5_train_from_scratch': True,
 'test_labels_paths_images_metrics_accessed_at_config_freeze': False,
}
if CONFIG_FREEZE_PATH.exists():
    frozen_config = json.loads(CONFIG_FREEZE_PATH.read_text())
    for key in ['EFFICIENTNET_CONFIG_FROZEN','monai_version','architecture','parameters','pretrained','seed','label_mapping','initial_lr','weight_decay','batch_size','precision','loss','sampler','augmentation','max_epochs','early_stop_patience','min_delta','fixed_test_membership_sha256','initialization_sha256']:
        assert frozen_config[key] == CONFIG[key], f'Frozen configuration mismatch: {key}'
    CONFIG = frozen_config
else:
    json_dump(CONFIG_FREEZE_PATH, CONFIG)
CONFIG_SHA256 = sha256_file(CONFIG_FREEZE_PATH)
print('EFFICIENTNET_CONFIG_FROZEN = TRUE')
print('Label mapping:', LABEL_MAP)
print('Architecture parameters:', f'{PARAMETERS:,}')
print('Initialization checksum:', INITIALIZATION_SHA256)
print('Physical GPU 2:', torch.cuda.get_device_name(0), GPU2_UUID)
print('Technical resume mode:', TECHNICAL_RESUME)
print('Fold 1 will be reused; Folds 2–5 will train from scratch. Fixed test remains sealed.')

EFFICIENTNET_CONFIG_FROZEN = TRUE
Label mapping: {'AD': 0, 'CN': 1, 'EMCI': 2, 'LMCI': 3, 'MCI': 4, 'SMC': 5}
Architecture parameters: 4,696,066
Initialization checksum: adec778a808e42db36ba24f9a7779b7f89e051b317586f039260c8d53b85acef
Physical GPU 2: NVIDIA GeForce RTX 3090 GPU-db657392-ab6d-94b9-02fd-9db3a5e31c3d
Technical resume mode: True
Fold 1 will be reused; Folds 2–5 will train from scratch. Fixed test remains sealed.


In [2]:
def class_auc_values(y_true, probabilities):
    return [float(roc_auc_score((np.asarray(y_true)==i).astype(int), probabilities[:,i])) for i in range(NUM_CLASSES)]

def gpu_snapshot(stage):
    out = subprocess.check_output(['nvidia-smi','--query-gpu=index,uuid,memory.used,utilization.gpu,power.draw,power.limit','--format=csv,noheader,nounits'], text=True)
    for line in out.splitlines():
        p=[x.strip() for x in line.split(',')]
        if p[1] == GPU2_UUID:
            snap={'stage':stage,'timestamp_utc':datetime.now(timezone.utc).isoformat(),'index':int(p[0]),'uuid':p[1],'memory_used_mib':int(float(p[2])),'utilization_percent':int(float(p[3])),'power_draw_w':float(p[4]),'power_limit_w':float(p[5]),'torch_allocated_mib':torch.cuda.memory_allocated()/2**20,'torch_reserved_mib':torch.cuda.memory_reserved()/2**20}
            return snap
    raise RuntimeError('Physical GPU 2 UUID missing')

def shutdown_loader(loader):
    iterator=getattr(loader,'_iterator',None)
    if iterator is not None:
        iterator._shutdown_workers()

def cleanup_gpu(stage, objects=None):
    if objects:
        for obj in objects:
            if isinstance(obj, DataLoader): shutdown_loader(obj)
    plt.close('all'); gc.collect(); torch.cuda.synchronize(); torch.cuda.empty_cache(); gc.collect()
    deadline=time.time()+90; samples=[]
    while True:
        snap=gpu_snapshot(stage); samples.append(snap)
        if snap['torch_allocated_mib'] < 128 and snap['torch_reserved_mib'] < 256 and snap['memory_used_mib'] < 1024:
            print(f"GPU cleanup {stage}: driver={snap['memory_used_mib']} MiB, torch allocated/reserved={snap['torch_allocated_mib']:.1f}/{snap['torch_reserved_mib']:.1f} MiB")
            return {'released':True,'release_criterion':'driver <1024 MiB, torch allocated <128 MiB, torch reserved <256 MiB; no multi-GB model footprint','single_notebook_context_retained':snap['memory_used_mib']>5,'samples':samples,'final':snap}
        if time.time() >= deadline: raise RuntimeError(f'GPU did not release after {stage}: {snap}')
        time.sleep(5)

def standardized_predictions(frame, metrics, fold):
    pos={str(x):i for i,x in enumerate(metrics['imageids'])}; order=np.array([pos[str(x)] for x in frame.ImageID])
    pred=metrics['predictions'][order]; prob=metrics['probabilities'][order]
    out=pd.DataFrame({'fold':fold,'ImageID':frame.ImageID.astype(str),'true_label':[CLASS_NAMES[i] for i in frame.GroupLabel.astype(int)],'predicted_label':[CLASS_NAMES[i] for i in pred]})
    for i,name in enumerate(CLASS_NAMES): out[f'probability {name}']=prob[:,i]
    out['correct']=out.true_label==out.predicted_label
    return out, frame.GroupLabel.astype(int).to_numpy(), pred, prob

def save_evaluation_artifacts(output_dir, split, y_true, predictions, probabilities, pred_frame):
    output_dir.mkdir(parents=True,exist_ok=True)
    pred_frame.to_csv(output_dir/f'{split}_predictions.csv',index=False)
    report=classification_report(y_true,predictions,labels=np.arange(NUM_CLASSES),target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    report_frame=pd.DataFrame(report).T.reset_index().rename(columns={'index':'class_or_average'})
    report_frame.to_csv(output_dir/f'classification_report_{split}.csv',index=False)
    (output_dir/f'classification_report_{split}.txt').write_text(classification_report(y_true,predictions,labels=np.arange(NUM_CLASSES),target_names=CLASS_NAMES,digits=6,zero_division=0),encoding='utf-8')
    cm=confusion_matrix(y_true,predictions,labels=np.arange(NUM_CLASSES)); norm=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
    pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(output_dir/f'confusion_matrix_{split}.csv')
    pd.DataFrame(norm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(output_dir/f'confusion_matrix_{split}_normalized.csv')
    for values,normalized,filename in [(cm,False,f'confusion_matrix_{split}.png'),(norm,True,f'confusion_matrix_{split}_normalized.png')]:
        fig,ax=plt.subplots(figsize=(7,6)); sns.heatmap(values,annot=True,fmt='.2f' if normalized else 'd',cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax)
        ax.set(xlabel='Predicted label',ylabel='True label',title=f"EfficientNet-B0 {split} {'normalized ' if normalized else ''}confusion matrix")
        fig.tight_layout(); fig.savefig(output_dir/filename,dpi=PLOT_DPI,bbox_inches='tight'); plt.close(fig)
    roc_rows=[]; fig,ax=plt.subplots(figsize=(7,6)); aucs=class_auc_values(y_true,probabilities)
    for i,name in enumerate(CLASS_NAMES):
        binary=(np.asarray(y_true)==i).astype(int); fpr,tpr,thresholds=roc_curve(binary,probabilities[:,i]); ax.plot(fpr,tpr,label=f'{name} (AUC={aucs[i]:.3f})')
        roc_rows.extend({'class':name,'fpr':float(x),'tpr':float(y),'threshold':float(z)} for x,y,z in zip(fpr,tpr,thresholds))
    pd.DataFrame(roc_rows).to_csv(output_dir/f'roc_{split}_data.csv',index=False)
    ax.plot([0,1],[0,1],'k--',alpha=.5); ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'EfficientNet-B0 {split} ROC'); ax.legend(loc='lower right',fontsize=8); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(output_dir/f'roc_{split}.png',dpi=PLOT_DPI,bbox_inches='tight'); plt.close(fig)
    precision,recall,f1,support=precision_recall_fscore_support(y_true,predictions,labels=np.arange(NUM_CLASSES),zero_division=0)
    per_class=pd.DataFrame({'class':CLASS_NAMES,'precision':precision,'recall':recall,'f1':f1,'auc':aucs,'support':support,'predicted_count':np.bincount(predictions,minlength=NUM_CLASSES)})
    per_class.to_csv(output_dir/f'per_class_metrics_{split}.csv',index=False)
    return per_class, report_frame

def save_learning_curves(history, output_dir):
    fig,axes=plt.subplots(2,3,figsize=(17,10)); axes=axes.ravel()
    for c,l in [('train_loss','Train'),('val_loss','Validation')]: axes[0].plot(history.epoch,history[c],label=l)
    for c,l in [('train_macro_f1','Train'),('val_macro_f1','Validation')]: axes[1].plot(history.epoch,history[c],label=l)
    axes[2].plot(history.epoch,history.val_macro_auc); axes[3].plot(history.epoch,history.val_mcc); axes[4].plot(history.epoch,history.learning_rate); axes[5].plot(history.epoch,history.train_val_macro_f1_gap)
    labels=[('Loss','Cross entropy'),('Macro F1','Macro F1'),('Validation Macro AUC','Macro AUC'),('Validation MCC','MCC'),('Learning rate','LR'),('Generalization gap','Train − validation Macro F1')]
    for ax,(title,ylabel) in zip(axes,labels): ax.set(title=title,xlabel='Epoch',ylabel=ylabel); ax.grid(alpha=.25); ax.legend(fontsize=8) if ax.get_legend_handles_labels()[0] else None
    fig.tight_layout(); fig.savefig(output_dir/'learning_curves.png',dpi=PLOT_DPI,bbox_inches='tight'); plt.close(fig)

def validate_split(fold, open_full=False):
    fold_dir=FOLDS_ROOT/f'fold_{fold}'; train_csv=fold_dir/'train.csv'; val_csv=fold_dir/'val.csv'; test_csv=fold_dir/'test.csv'
    assert sha256_file(train_csv)==SPLIT_HASHES[fold]['train'] and sha256_file(val_csv)==SPLIT_HASHES[fold]['val'] and sha256_file(test_csv)==EXPECTED_TEST_SHA256
    train_df=read_split(train_csv); val_df=read_split(val_csv); test_ids=pd.read_csv(test_csv,usecols=['ImageID'],dtype={'ImageID':str}).ImageID
    tr,va,te=set(train_df.ImageID),set(val_df.ImageID),set(test_ids); expected_train,expected_val=EXPECTED_ROWS[fold]
    assert len(train_df)==expected_train and len(val_df)==expected_val and len(test_ids)==719 and not(tr&va) and not(tr&te) and not(va&te)
    return train_df,val_df,test_ids

def metric_row(fold, metrics):
    return {'Fold':fold,'Accuracy':metrics['accuracy'],'Macro Precision':metrics['macro_precision'],'Macro Recall':metrics['macro_recall'],'Macro F1':metrics['macro_f1'],'MCC':metrics['mcc'],'Macro AUC':metrics['macro_auc']}

def reusable_fold1():
    out=RESULTS_ROOT/'fold_1'; out.mkdir()
    for name in ['best_model.pt','latest_checkpoint.pt','history.csv','per_class_history.csv','health_checks.json','runtime_summary.json','model_complexity.csv']:
        shutil.copy2(FOLD1_SOURCE/name,out/name)
    train_df,val_df,test_ids=validate_split(1)
    source_pred=pd.read_csv(FOLD1_SOURCE/'validation_predictions.csv',dtype={'ImageID':str}); pos={str(x):i for i,x in enumerate(source_pred.ImageID)}; order=np.array([pos[str(x)] for x in val_df.ImageID])
    prob=source_pred[[f'prob_{n}' for n in CLASS_NAMES]].to_numpy()[order]; y_true=val_df.GroupLabel.astype(int).to_numpy(); metrics=calculate_metrics(y_true,prob); metrics['imageids']=val_df.ImageID.astype(str).tolist()
    pred_frame,y_true,predictions,probabilities=standardized_predictions(val_df,metrics,1); per_class,_=save_evaluation_artifacts(out,'validation',y_true,predictions,probabilities,pred_frame)
    source_metrics=pd.read_csv(FOLD1_SOURCE/'validation_metrics.csv').iloc[0].to_dict(); pd.DataFrame([source_metrics]).to_csv(out/'validation_metrics.csv',index=False)
    pd.DataFrame([{'split':'train','rows':len(train_df),'unique_imageids':train_df.ImageID.nunique(),'csv_sha256':SPLIT_HASHES[1]['train'],'exact_persisted_assignment':True},{'split':'validation','rows':len(val_df),'unique_imageids':val_df.ImageID.nunique(),'csv_sha256':SPLIT_HASHES[1]['val'],'exact_persisted_assignment':True}]).to_csv(out/'fold_split_audit.csv',index=False)
    shutil.copy2(FOLD1_SOURCE/'learning_curves.png',out/'learning_curves.png')
    json_dump(out/'config.json',{'fold':1,'validated_fold1_reused':True,'source':str(FOLD1_SOURCE),'source_checkpoint_sha256':sha256_file(FOLD1_SOURCE/'best_model.pt'),'configuration_sha256':CONFIG_SHA256,'test_inference_performed':False})
    json_dump(out/'integrity_complete.json',{'valid':True,'fold':1,'checkpoint_reused_without_retraining':True,'all_six_classes_predicted':True,'test_inference_performed':False})
    return source_metrics,per_class

def completed_fold_available(fold):
    out=RESULTS_ROOT/f'fold_{fold}'
    required=['best_model.pt','history.csv','validation_metrics.csv','per_class_metrics_validation.csv','config.json','integrity_complete.json']
    if not all((out/name).is_file() for name in required): return False
    integrity=json.loads((out/'integrity_complete.json').read_text()); config=json.loads((out/'config.json').read_text())
    return bool(integrity.get('valid')) and config.get('configuration_sha256')==CONFIG_SHA256

def load_completed_fold(fold):
    out=RESULTS_ROOT/f'fold_{fold}'; assert completed_fold_available(fold)
    validate_split(fold)
    metrics=pd.read_csv(out/'validation_metrics.csv').iloc[0].to_dict(); per_class=pd.read_csv(out/'per_class_metrics_validation.csv')
    checkpoint=torch.load(out/'best_model.pt',map_location='cpu',weights_only=False)
    if fold>1:
        assert checkpoint['configuration_sha256']==CONFIG_SHA256 and checkpoint['initialization_sha256']==INITIALIZATION_SHA256
        assert abs(float(checkpoint['best_val_macro_f1'])-float(metrics['Macro F1']))<1e-10
        history=pd.read_csv(out/'history.csv')
        print(f'Fold {fold} technical resume: replaying {len(history)} recorded epochs from the completed, integrity-checked scratch run.')
        for _,r in history.iterrows():
            print(f"Fold {fold} Epoch {int(r.epoch):02d}/{MAX_EPOCHS} | loss={r.train_loss:.4f}/{r.val_loss:.4f} | acc={r.train_accuracy:.4f}/{r.val_accuracy:.4f} | P={r.train_macro_precision:.4f}/{r.val_macro_precision:.4f} | R={r.train_macro_recall:.4f}/{r.val_macro_recall:.4f} | F1={r.train_macro_f1:.4f}/{r.val_macro_f1:.4f} | MCC={r.val_mcc:.4f} AUC={r.val_macro_auc:.4f} | classes={int(r.validation_predicted_classes)}/6 | gap={r.train_val_macro_f1_gap:.4f} lr={r.learning_rate:.2e} time={r.epoch_minutes:.2f}m [RECORDED]")
        del history
    del checkpoint
    release=cleanup_gpu(f'after integrity-checked technical resume Fold {fold}'); json_dump(out/'gpu_release_after_validation.json',release)
    return metrics,per_class,release

def train_fold(fold):
    out=RESULTS_ROOT/f'fold_{fold}'; out.mkdir(); train_df,val_df,test_ids=validate_split(fold)
    pd.DataFrame([{'split':'train','rows':len(train_df),'unique_imageids':train_df.ImageID.nunique(),'csv_sha256':SPLIT_HASHES[fold]['train'],'exact_persisted_assignment':True},{'split':'validation','rows':len(val_df),'unique_imageids':val_df.ImageID.nunique(),'csv_sha256':SPLIT_HASHES[fold]['val'],'exact_persisted_assignment':True}]).to_csv(out/'fold_split_audit.csv',index=False)
    train_ds,val_ds=Fold1MRIDataset(train_df),Fold1MRIDataset(val_df); generator=torch.Generator().manual_seed(SEED)
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,generator=generator,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True,prefetch_factor=PREFETCH_FACTOR,worker_init_fn=seed_worker)
    val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True,prefetch_factor=PREFETCH_FACTOR,worker_init_fn=seed_worker)
    assert isinstance(train_loader.sampler,RandomSampler)
    seed_all(SEED); model=build_model('E'); init_sha=state_checksum(model); assert init_sha==INITIALIZATION_SHA256; model=model.cuda()
    dummy=torch.zeros(1,1,128,128,128,device='cuda'); model.eval()
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16): dummy_out=model(dummy)
    assert tuple(dummy_out.shape)==(1,6) and torch.isfinite(dummy_out).all(); del dummy,dummy_out,model; gc.collect(); torch.cuda.empty_cache()
    seed_all(SEED); model=build_model('E'); assert state_checksum(model)==init_sha; model=model.cuda(); criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=INITIAL_LR,weight_decay=WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=1/3,patience=3)
    history_rows=[]; per_class_rows=[]; health=[]; best_f1=-np.inf; best_epoch=None; patience=0; started=time.time(); torch.cuda.reset_peak_memory_stats()
    print(f'\nFOLD {fold} TRAINING START — fresh scratch initialization {init_sha}')
    for epoch in range(1,MAX_EPOCHS+1):
        epoch_started=time.time(); train_metrics=run_phase(model,train_loader,criterion,True,optimizer,torch.device('cuda:0'),epoch)
        with torch.no_grad(): val_metrics=run_phase(model,val_loader,criterion,False,None,torch.device('cuda:0'),epoch)
        minutes=(time.time()-epoch_started)/60; predicted=val_metrics['predicted_counts']; classes=int(np.count_nonzero(predicted)); dominant=float(predicted.max()/predicted.sum())
        if dominant>.90: health.append({'epoch':epoch,'warning':'prediction concentration >90%'})
        if epoch>=10 and classes<6: health.append({'epoch':epoch,'warning':'fewer than six predicted classes'})
        row={'epoch':epoch,'train_loss':train_metrics['loss'],'val_loss':val_metrics['loss'],'train_accuracy':train_metrics['accuracy'],'val_accuracy':val_metrics['accuracy'],'train_macro_precision':train_metrics['macro_precision'],'val_macro_precision':val_metrics['macro_precision'],'train_macro_recall':train_metrics['macro_recall'],'val_macro_recall':val_metrics['macro_recall'],'train_macro_f1':train_metrics['macro_f1'],'val_macro_f1':val_metrics['macro_f1'],'val_mcc':val_metrics['mcc'],'val_macro_auc':val_metrics['macro_auc'],'train_val_macro_f1_gap':train_metrics['macro_f1']-val_metrics['macro_f1'],'learning_rate':optimizer.param_groups[0]['lr'],'validation_predicted_classes':classes,'dominant_prediction_fraction':dominant,'epoch_minutes':minutes}
        for i,name in enumerate(CLASS_NAMES):
            cr={'epoch':epoch,'class':name,'precision':float(val_metrics['precision'][i]),'recall':float(val_metrics['recall'][i]),'f1':float(val_metrics['f1'][i]),'support':int(val_metrics['support'][i]),'predicted_count':int(predicted[i])}; per_class_rows.append(cr)
            for key in ['precision','recall','f1','support','predicted_count']: row[f'val_{name}_{key}']=cr[key]
        history_rows.append(row); pd.DataFrame(history_rows).to_csv(out/'history.csv',index=False); pd.DataFrame(per_class_rows).to_csv(out/'per_class_history.csv',index=False)
        improved=np.isfinite(val_metrics['macro_f1']) and val_metrics['macro_f1']>best_f1+MIN_DELTA
        if improved:
            best_f1=val_metrics['macro_f1']; best_epoch=epoch; patience=0; torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'best_val_macro_f1':best_f1,'checkpoint_selection':'validation_macro_f1_only','fold':fold,'architecture':'MONAI EfficientNet-B0 3D','pretrained':False,'initialization_sha256':init_sha,'configuration_sha256':CONFIG_SHA256,'test_inference_performed':False},out/'best_model.pt')
        else: patience+=1
        scheduler.step(val_metrics['macro_f1']); torch.save({'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),'scheduler_state_dict':scheduler.state_dict(),'epoch':epoch,'best_epoch':best_epoch,'best_val_macro_f1':best_f1,'resume_default':False},out/'latest_checkpoint.pt')
        print(f"Fold {fold} Epoch {epoch:02d}/{MAX_EPOCHS} | loss={train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | acc={train_metrics['accuracy']:.4f}/{val_metrics['accuracy']:.4f} | P={train_metrics['macro_precision']:.4f}/{val_metrics['macro_precision']:.4f} | R={train_metrics['macro_recall']:.4f}/{val_metrics['macro_recall']:.4f} | F1={train_metrics['macro_f1']:.4f}/{val_metrics['macro_f1']:.4f} | MCC={val_metrics['mcc']:.4f} AUC={val_metrics['macro_auc']:.4f} | classes={classes}/6 counts={predicted.tolist()} | gap={row['train_val_macro_f1_gap']:.4f} lr={row['learning_rate']:.2e} time={minutes:.2f}m")
        if patience>=EARLY_STOP_PATIENCE: print(f'Early stopping Fold {fold} at epoch {epoch}; best={best_epoch}, Macro F1={best_f1:.6f}'); break
    training_minutes=(time.time()-started)/60; history=pd.DataFrame(history_rows); checkpoint=torch.load(out/'best_model.pt',map_location='cuda',weights_only=False); model.load_state_dict(checkpoint['model_state_dict'],strict=True)
    with torch.no_grad(): metrics=run_phase(model,val_loader,criterion,False,None,torch.device('cuda:0'),best_epoch)
    assert abs(metrics['macro_f1']-checkpoint['best_val_macro_f1'])<1e-10 and np.count_nonzero(metrics['predicted_counts'])==6
    pred_frame,y_true,predictions,probabilities=standardized_predictions(val_df,metrics,fold); per_class,_=save_evaluation_artifacts(out,'validation',y_true,predictions,probabilities,pred_frame)
    best_row=history.loc[history.epoch==best_epoch].iloc[0]; vm={**metric_row(fold,metrics),'best_epoch':int(best_epoch),'train_macro_f1_at_best_epoch':float(best_row.train_macro_f1),'train_validation_macro_f1_gap':float(best_row.train_val_macro_f1_gap),'predicted_classes':6,'test_inference_performed':False}
    pd.DataFrame([vm]).to_csv(out/'validation_metrics.csv',index=False); save_learning_curves(history,out)
    runtime={'fold':fold,'valid':True,'precision_mode':'bf16','gpu':torch.cuda.get_device_name(0),'physical_gpu_uuid':GPU2_UUID,'epochs_completed':int(history.epoch.max()),'best_epoch':int(best_epoch),'average_epoch_minutes':float(history.epoch_minutes.mean()),'total_training_minutes':training_minutes,'peak_gpu_memory_gib':torch.cuda.max_memory_allocated()/2**30,'parameters':PARAMETERS,'checkpoint_size_mib':(out/'best_model.pt').stat().st_size/2**20}
    json_dump(out/'runtime_summary.json',runtime); pd.DataFrame([runtime]).to_csv(out/'model_complexity.csv',index=False); json_dump(out/'health_checks.json',{'valid':True,'events':health,'finite_inputs_logits_losses_gradients':True,'technical_collapse':False,'all_six_classes_at_checkpoint':True})
    json_dump(out/'config.json',{'fold':fold,'fresh_scratch':True,'initialization_sha256':init_sha,'configuration_sha256':CONFIG_SHA256,'test_inference_performed':False}); json_dump(out/'integrity_complete.json',{'valid':True,'fold':fold,'checkpoint_reloaded':True,'checkpoint_macro_f1_reproduced':True,'all_six_classes_predicted':True,'test_inference_performed':False})
    shutdown_loader(train_loader); shutdown_loader(val_loader); del checkpoint,model,optimizer,scheduler,criterion,train_loader,val_loader,train_ds,val_ds,train_df,val_df,pred_frame,predictions,probabilities,y_true,metrics,train_metrics,val_metrics,best_row,generator
    release=cleanup_gpu(f'after validation Fold {fold}'); json_dump(out/'gpu_release_after_validation.json',release)
    return vm,per_class


In [3]:
validation_records={}; validation_classwise={}; gpu_audit={'one_notebook_only':True,'training_order':FOLDS,'test_order':FOLDS,'physical_gpu_uuid':GPU2_UUID,'fold1_reused':True,'technical_resume':TECHNICAL_RESUME,'events':[]}
for fold in [1, 2, 3, 4, 5]:
    print(f'\n===== EFFICIENTNET-B0 FOLD {fold} =====')
    if completed_fold_available(fold):
        metrics,per_class,release=load_completed_fold(fold)
        print(f'Fold {fold} reused after artifact/configuration integrity checks:',pd.DataFrame([metrics]).to_string(index=False))
    elif fold == 1:
        metrics,per_class=reusable_fold1(); release=cleanup_gpu('after reused Fold 1'); json_dump(RESULTS_ROOT/'fold_1'/'gpu_release_after_validation.json',release)
        print('Fold 1 reused without retraining:',pd.DataFrame([metrics]).to_string(index=False))
    else:
        metrics,per_class=train_fold(fold)
        release=json.loads((RESULTS_ROOT/f'fold_{fold}'/'gpu_release_after_validation.json').read_text())
    validation_records[fold]=metrics; validation_classwise[fold]=per_class
    gpu_audit['events'].append({'phase':'validation','fold':fold,'release':release['final']})
    json_dump(SUMMARY_DIR/'EfficientNet_B0_GPU_Sequential_Audit.json',gpu_audit)
print('\nALL FIVE VALIDATION FOLDS COMPLETE; fixed test has not been evaluated.')


===== EFFICIENTNET-B0 FOLD 1 =====


GPU cleanup after integrity-checked technical resume Fold 1: driver=263 MiB, torch allocated/reserved=0.0/0.0 MiB
Fold 1 reused after artifact/configuration integrity checks:  best_epoch  accuracy  macro_precision  macro_recall  macro_f1      mcc  macro_auc  train_macro_f1_at_best_epoch  train_validation_macro_f1_gap  predicted_classes  test_inference_performed
         33  0.937391          0.92303      0.909931  0.915892 0.920234   0.988291                      0.998788                       0.082896                  6                     False

===== EFFICIENTNET-B0 FOLD 2 =====


Fold 2 technical resume: replaying 35 recorded epochs from the completed, integrity-checked scratch run.
Fold 2 Epoch 01/40 | loss=2.4582/1.7523 | acc=0.2328/0.3043 | P=0.1811/0.0507 | R=0.1731/0.1667 | F1=0.1726/0.0778 | MCC=0.0000 AUC=0.5390 | classes=1/6 | gap=0.0948 lr=1.00e-04 time=1.48m [RECORDED]
Fold 2 Epoch 02/40 | loss=1.9559/1.5673 | acc=0.2607/0.3687 | P=0.1897/0.3904 | R=0.1862/0.2738 | F1=0.1830/0.2246 | MCC=0.1884 AUC=0.6815 | classes=5/6 | gap=-0.0416 lr=1.00e-04 time=1.47m [RECORDED]
Fold 2 Epoch 03/40 | loss=1.5554/1.9251 | acc=0.4225/0.4052 | P=0.3640/0.4325 | R=0.3316/0.2873 | F1=0.3339/0.2376 | MCC=0.2711 AUC=0.8145 | classes=4/6 | gap=0.0963 lr=1.00e-04 time=1.45m [RECORDED]
Fold 2 Epoch 04/40 | loss=1.0534/1.8203 | acc=0.6323/0.5043 | P=0.5751/0.5775 | R=0.5332/0.4192 | F1=0.5352/0.3257 | MCC=0.3986 AUC=0.8666 | classes=6/6 | gap=0.2095 lr=1.00e-04 time=1.46m [RECORDED]
Fold 2 Epoch 05/40 | loss=0.7928/1.0785 | acc=0.7594/0.6557 | P=0.7241/0.7102 | R=0.6628/0.603

GPU cleanup after integrity-checked technical resume Fold 2: driver=263 MiB, torch allocated/reserved=0.0/0.0 MiB
Fold 2 reused after artifact/configuration integrity checks:  Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC  best_epoch  train_macro_f1_at_best_epoch  train_validation_macro_f1_gap  predicted_classes  test_inference_performed
    2  0.932174         0.940968      0.916403  0.927221 0.913668   0.982573          25                      0.993876                       0.066656                  6                     False

===== EFFICIENTNET-B0 FOLD 3 =====


Fold 3 technical resume: replaying 33 recorded epochs from the completed, integrity-checked scratch run.
Fold 3 Epoch 01/40 | loss=2.4060/1.6666 | acc=0.2611/0.3026 | P=0.1967/0.0504 | R=0.1932/0.1667 | F1=0.1913/0.0774 | MCC=0.0000 AUC=0.5205 | classes=1/6 | gap=0.1139 lr=1.00e-04 time=1.53m [RECORDED]
Fold 3 Epoch 02/40 | loss=1.9351/1.6275 | acc=0.2842/0.3339 | P=0.1978/0.1885 | R=0.1978/0.2417 | F1=0.1918/0.1859 | MCC=0.1229 AUC=0.6290 | classes=4/6 | gap=0.0059 lr=1.00e-04 time=1.46m [RECORDED]
Fold 3 Epoch 03/40 | loss=1.6481/1.3136 | acc=0.3599/0.5183 | P=0.2887/0.5021 | R=0.2748/0.3702 | F1=0.2749/0.3674 | MCC=0.3699 AUC=0.7843 | classes=5/6 | gap=-0.0925 lr=1.00e-04 time=1.40m [RECORDED]
Fold 3 Epoch 04/40 | loss=1.1017/0.9122 | acc=0.6218/0.6748 | P=0.5491/0.6276 | R=0.5101/0.5440 | F1=0.5129/0.5420 | MCC=0.5916 AUC=0.8979 | classes=5/6 | gap=-0.0291 lr=1.00e-04 time=1.41m [RECORDED]
Fold 3 Epoch 05/40 | loss=0.7711/1.0883 | acc=0.7550/0.7200 | P=0.7029/0.7611 | R=0.6529/0.64

GPU cleanup after integrity-checked technical resume Fold 3: driver=263 MiB, torch allocated/reserved=0.0/0.0 MiB
Fold 3 reused after artifact/configuration integrity checks:  Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC  best_epoch  train_macro_f1_at_best_epoch  train_validation_macro_f1_gap  predicted_classes  test_inference_performed
    3      0.92         0.894173       0.86488  0.878226 0.897955   0.962749          23                      0.994816                       0.116589                  6                     False

===== EFFICIENTNET-B0 FOLD 4 =====


Fold 4 technical resume: replaying 40 recorded epochs from the completed, integrity-checked scratch run.
Fold 4 Epoch 01/40 | loss=2.4271/1.7885 | acc=0.2532/0.3031 | P=0.1889/0.0505 | R=0.1863/0.1667 | F1=0.1849/0.0775 | MCC=0.0000 AUC=0.5269 | classes=1/6 | gap=0.1074 lr=1.00e-04 time=1.53m [RECORDED]
Fold 4 Epoch 02/40 | loss=1.8914/1.5530 | acc=0.2853/0.3502 | P=0.2049/0.2790 | R=0.2038/0.2294 | F1=0.1988/0.1593 | MCC=0.1641 AUC=0.6850 | classes=4/6 | gap=0.0396 lr=1.00e-04 time=1.47m [RECORDED]
Fold 4 Epoch 03/40 | loss=1.5042/1.9615 | acc=0.4302/0.3537 | P=0.3448/0.6084 | R=0.3380/0.2195 | F1=0.3367/0.1759 | MCC=0.1828 AUC=0.7963 | classes=5/6 | gap=0.1608 lr=1.00e-04 time=1.44m [RECORDED]
Fold 4 Epoch 04/40 | loss=0.9719/0.8572 | acc=0.6699/0.7300 | P=0.6041/0.8014 | R=0.5643/0.5977 | F1=0.5650/0.5942 | MCC=0.6545 AUC=0.9285 | classes=6/6 | gap=-0.0292 lr=1.00e-04 time=1.44m [RECORDED]
Fold 4 Epoch 05/40 | loss=0.7163/0.8495 | acc=0.7829/0.7909 | P=0.7299/0.6614 | R=0.6922/0.662

GPU cleanup after integrity-checked technical resume Fold 4: driver=263 MiB, torch allocated/reserved=0.0/0.0 MiB
Fold 4 reused after artifact/configuration integrity checks:  Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC  best_epoch  train_macro_f1_at_best_epoch  train_validation_macro_f1_gap  predicted_classes  test_inference_performed
    4  0.937282         0.927407      0.924372   0.92531 0.920291   0.979189          39                      0.998855                       0.073545                  6                     False

===== EFFICIENTNET-B0 FOLD 5 =====


Fold 5 technical resume: replaying 40 recorded epochs from the completed, integrity-checked scratch run.
Fold 5 Epoch 01/40 | loss=2.4052/1.7316 | acc=0.2410/0.2596 | P=0.1683/0.0433 | R=0.1725/0.1667 | F1=0.1691/0.0687 | MCC=0.0000 AUC=0.5469 | classes=1/6 | gap=0.1004 lr=1.00e-04 time=1.55m [RECORDED]
Fold 5 Epoch 02/40 | loss=1.9136/1.6113 | acc=0.2745/0.3519 | P=0.2011/0.2975 | R=0.1954/0.2143 | F1=0.1923/0.1559 | MCC=0.1412 AUC=0.7016 | classes=4/6 | gap=0.0364 lr=1.00e-04 time=1.48m [RECORDED]
Fold 5 Epoch 03/40 | loss=1.4049/0.9243 | acc=0.4772/0.6638 | P=0.3739/0.6168 | R=0.3696/0.5240 | F1=0.3672/0.4920 | MCC=0.5759 AUC=0.9018 | classes=5/6 | gap=-0.1248 lr=1.00e-04 time=1.46m [RECORDED]
Fold 5 Epoch 04/40 | loss=0.9834/0.9289 | acc=0.6786/0.6899 | P=0.6036/0.6109 | R=0.5657/0.5991 | F1=0.5692/0.5672 | MCC=0.6269 AUC=0.9324 | classes=5/6 | gap=0.0020 lr=1.00e-04 time=1.44m [RECORDED]
Fold 5 Epoch 05/40 | loss=0.7655/0.8530 | acc=0.7751/0.7962 | P=0.7358/0.6663 | R=0.6824/0.685

GPU cleanup after integrity-checked technical resume Fold 5: driver=263 MiB, torch allocated/reserved=0.0/0.0 MiB
Fold 5 reused after artifact/configuration integrity checks:  Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC  best_epoch  train_macro_f1_at_best_epoch  train_validation_macro_f1_gap  predicted_classes  test_inference_performed
    5  0.940767         0.917802      0.912482  0.913194 0.925076   0.963636          40                      0.995212                       0.082017                  6                     False

ALL FIVE VALIDATION FOLDS COMPLETE; fixed test has not been evaluated.


In [4]:
METRICS=['Accuracy','Macro Precision','Macro Recall','Macro F1','MCC','Macro AUC']
validation_rows=[]
for fold in FOLDS:
    m=validation_records[fold]; validation_rows.append({'Fold':f'Fold {fold}','Accuracy':float(m.get('Accuracy',m.get('accuracy'))),'Macro Precision':float(m.get('Macro Precision',m.get('macro_precision'))),'Macro Recall':float(m.get('Macro Recall',m.get('macro_recall'))),'Macro F1':float(m.get('Macro F1',m.get('macro_f1'))),'MCC':float(m.get('MCC',m.get('mcc'))),'Macro AUC':float(m.get('Macro AUC',m.get('macro_auc')))})
validation_fold_frame=pd.DataFrame(validation_rows)
validation_summary=pd.concat([validation_fold_frame,pd.DataFrame([{'Fold':'Mean',**{m:validation_fold_frame[m].mean() for m in METRICS}},{'Fold':'SD',**{m:validation_fold_frame[m].std(ddof=1) for m in METRICS}}])],ignore_index=True)
validation_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Fold_Validation_Summary.csv',index=False)
classwise_rows=[]
for name in CLASS_NAMES:
    row={'Class':name}
    for measure in ['precision','recall','f1','auc']:
        values=[]
        for fold in FOLDS:
            value=float(validation_classwise[fold].loc[validation_classwise[fold]['class']==name,measure].iloc[0]); values.append(value); row[f'Fold {fold} {measure.title()}']=value
        row[f'Mean {measure.title()}']=np.mean(values); row[f'SD {measure.title()}']=np.std(values,ddof=1)
    classwise_rows.append(row)
validation_classwise_summary=pd.DataFrame(classwise_rows); validation_classwise_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Fold_Classwise_Validation_Summary.csv',index=False)
smc_validation=[]
for fold in FOLDS:
    r=validation_classwise[fold].set_index('class').loc['SMC']; smc_validation.append({'Fold':f'Fold {fold}','Support':float(r.support),'Precision':float(r.precision),'Recall':float(r.recall),'F1':float(r.f1),'AUC':float(r.auc)})
smc_frame=pd.DataFrame(smc_validation); smc_summary=pd.concat([smc_frame,pd.DataFrame([{'Fold':'Mean',**{c:smc_frame[c].mean() for c in ['Support','Precision','Recall','F1','AUC']}},{'Fold':'SD',**{c:smc_frame[c].std(ddof=1) for c in ['Support','Precision','Recall','F1','AUC']}}])],ignore_index=True); smc_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_SMC_Validation_Summary.csv',index=False)
checkpoint_freeze={'CHECKPOINTS_FROZEN_BEFORE_TEST':True,'freeze_timestamp_utc':datetime.now(timezone.utc).isoformat(),'configuration_path':str(CONFIG_FREEZE_PATH),'configuration_sha256':CONFIG_SHA256,'validation_summary_path':str(SUMMARY_DIR/'EfficientNet_B0_5Fold_Validation_Summary.csv'),'validation_summary_sha256':sha256_file(SUMMARY_DIR/'EfficientNet_B0_5Fold_Validation_Summary.csv'),'checkpoints':{}}
for fold in FOLDS:
    p=RESULTS_ROOT/f'fold_{fold}'/'best_model.pt'; checkpoint_freeze['checkpoints'][f'Fold {fold}']={'path':str(p),'sha256':sha256_file(p)}
json_dump(SUMMARY_DIR/'EfficientNet_B0_Final_Checkpoint_Freeze.json',checkpoint_freeze)
print(validation_summary.to_string(index=False))
print('\nCHECKPOINTS_FROZEN_BEFORE_TEST = TRUE')

  Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
Fold 1  0.937391         0.923030      0.909931  0.915892 0.920234   0.988291
Fold 2  0.932174         0.940968      0.916403  0.927221 0.913668   0.982573
Fold 3  0.920000         0.894173      0.864880  0.878226 0.897955   0.962749
Fold 4  0.937282         0.927407      0.924372  0.925310 0.920291   0.979189
Fold 5  0.940767         0.917802      0.912482  0.913194 0.925076   0.963636
  Mean  0.933523         0.920676      0.905613  0.911969 0.915445   0.975288
    SD  0.008159         0.017127      0.023416  0.019787 0.010586   0.011515

CHECKPOINTS_FROZEN_BEFORE_TEST = TRUE


In [5]:
freeze=json.loads((SUMMARY_DIR/'EfficientNet_B0_Final_Checkpoint_Freeze.json').read_text()); assert freeze['CHECKPOINTS_FROZEN_BEFORE_TEST'] is True and CONFIG['EFFICIENTNET_CONFIG_FROZEN'] is True
test_csv=FOLDS_ROOT/'fold_1'/'test.csv'; assert sha256_file(test_csv)==EXPECTED_TEST_SHA256
test_df=read_split(test_csv); assert len(test_df)==719 and test_df.ImageID.nunique()==719
test_records={}; test_classwise={}; all_test_predictions=[]; test_runtime=[]
for fold in [1, 2, 3, 4, 5]:
    print(f'\n===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD {fold} =====')
    out=RESULTS_ROOT/f'fold_{fold}'; checkpoint_path=out/'best_model.pt'; expected=freeze['checkpoints'][f'Fold {fold}']['sha256']; assert sha256_file(checkpoint_path)==expected
    seed_all(SEED); model=build_model('E'); assert state_checksum(model)==INITIALIZATION_SHA256; checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=False); model.load_state_dict(checkpoint['model_state_dict'],strict=True); model=model.cuda(); criterion=nn.CrossEntropyLoss()
    test_ds=Fold1MRIDataset(test_df); test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True,prefetch_factor=PREFETCH_FACTOR,worker_init_fn=seed_worker)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); started=time.time()
    with torch.no_grad(): metrics=run_phase(model,test_loader,criterion,False,None,torch.device('cuda:0'),f'test-{fold}')
    torch.cuda.synchronize(); seconds=time.time()-started; pred_frame,y_true,predictions,probabilities=standardized_predictions(test_df,metrics,fold); per_class,_=save_evaluation_artifacts(out,'test',y_true,predictions,probabilities,pred_frame)
    row=metric_row(fold,metrics); pd.DataFrame([row]).to_csv(out/'test_metrics.csv',index=False); all_test_predictions.append(pred_frame); test_records[fold]=row; test_classwise[fold]=per_class
    rt={'fold':fold,'images':719,'inference_seconds':seconds,'inference_seconds_per_mri':seconds/719,'peak_gpu_memory_gib':torch.cuda.max_memory_allocated()/2**30}; test_runtime.append(rt); json_dump(out/'test_runtime.json',rt)
    print(pd.DataFrame([row]).to_string(index=False))
    shutdown_loader(test_loader); del checkpoint,model,criterion,test_loader,test_ds,pred_frame,predictions,probabilities,y_true,metrics,per_class
    release=cleanup_gpu(f'after test checkpoint Fold {fold}'); json_dump(out/'gpu_release_after_test.json',release); gpu_audit['events'].append({'phase':'test','fold':fold,'release':release['final']}); json_dump(SUMMARY_DIR/'EfficientNet_B0_GPU_Sequential_Audit.json',gpu_audit)
combined_test=pd.concat(all_test_predictions,ignore_index=True); assert len(combined_test)==3595 and combined_test.groupby('fold').size().eq(719).all() and combined_test.ImageID.nunique()==719
combined_test.to_csv(SUMMARY_DIR/'EfficientNet_B0_All_Test_Predictions.csv',index=False)
test_fold_frame=pd.DataFrame([test_records[f] for f in FOLDS]); test_summary=pd.concat([test_fold_frame,pd.DataFrame([{'Fold':'Mean',**{m:test_fold_frame[m].mean() for m in METRICS}},{'Fold':'SD',**{m:test_fold_frame[m].std(ddof=1) for m in METRICS}}])],ignore_index=True); test_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Fold_Test_Summary.csv',index=False)
test_classwise_rows=[]
for name in CLASS_NAMES:
    row={'Class':name}
    for measure in ['precision','recall','f1','auc']:
        values=[]
        for fold in FOLDS:
            value=float(test_classwise[fold].loc[test_classwise[fold]['class']==name,measure].iloc[0]); values.append(value); row[f'Fold {fold} {measure.title()}']=value
        row[f'Mean {measure.title()}']=np.mean(values); row[f'SD {measure.title()}']=np.std(values,ddof=1)
    test_classwise_rows.append(row)
test_classwise_summary=pd.DataFrame(test_classwise_rows); test_classwise_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Fold_Test_Classwise_Summary.csv',index=False)
print('\nFive independently trained fold checkpoints evaluated on the same fixed test cohort:')
print(test_summary.to_string(index=False))


===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD 1 =====


 Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
    1   0.93185         0.936664      0.895405  0.912942 0.913006   0.979423


GPU cleanup after test checkpoint Fold 1: driver=379 MiB, torch allocated/reserved=32.0/32.0 MiB

===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD 2 =====


 Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
    2  0.916551         0.920687       0.86389  0.886885 0.893449   0.972594


GPU cleanup after test checkpoint Fold 2: driver=379 MiB, torch allocated/reserved=32.0/32.0 MiB

===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD 3 =====


 Fold  Accuracy  Macro Precision  Macro Recall  Macro F1     MCC  Macro AUC
    3  0.916551         0.882502       0.87767  0.878843 0.89374   0.976329


GPU cleanup after test checkpoint Fold 3: driver=379 MiB, torch allocated/reserved=32.0/32.0 MiB

===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD 4 =====


 Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
    4   0.91516         0.889051      0.879938  0.883302 0.891791   0.968833


GPU cleanup after test checkpoint Fold 4: driver=379 MiB, torch allocated/reserved=32.0/32.0 MiB

===== FIXED TEST: EFFICIENTNET CHECKPOINT FOLD 5 =====


 Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
    5  0.934631         0.940917      0.882137    0.9025 0.916642   0.975929


GPU cleanup after test checkpoint Fold 5: driver=379 MiB, torch allocated/reserved=32.0/32.0 MiB

Five independently trained fold checkpoints evaluated on the same fixed test cohort:
Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC
   1  0.931850         0.936664      0.895405  0.912942 0.913006   0.979423
   2  0.916551         0.920687      0.863890  0.886885 0.893449   0.972594
   3  0.916551         0.882502      0.877670  0.878843 0.893740   0.976329
   4  0.915160         0.889051      0.879938  0.883302 0.891791   0.968833
   5  0.934631         0.940917      0.882137  0.902500 0.916642   0.975929
Mean  0.922949         0.913964      0.879808  0.892895 0.901726   0.974621
  SD  0.009464         0.026914      0.011255  0.014315 0.012049   0.004040


In [6]:
truth_check=combined_test.groupby('ImageID').true_label.nunique(); assert truth_check.eq(1).all()
metadata=combined_test.sort_values('fold').drop_duplicates('ImageID')[['ImageID','true_label']]; means=combined_test.groupby('ImageID',as_index=False)[PROBABILITY_COLUMNS].mean(); ensemble=metadata.merge(means,on='ImageID',validate='one_to_one')
y_true=np.array([LABEL_MAP[x] for x in ensemble.true_label]); probabilities=ensemble[PROBABILITY_COLUMNS].to_numpy(); metrics=calculate_metrics(y_true,probabilities); ensemble['predicted_label']=[CLASS_NAMES[i] for i in metrics['predictions']]; ensemble['correct']=ensemble.true_label==ensemble.predicted_label; ensemble['analysis']='Secondary ensemble analysis'; ensemble.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_Predictions.csv',index=False)
ensemble_metrics=pd.DataFrame([{**metric_row('Five-model probability ensemble',metrics),'Analysis':'Secondary ensemble analysis'}]); ensemble_metrics.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_Metrics.csv',index=False)
report=classification_report(y_true,metrics['predictions'],labels=np.arange(NUM_CLASSES),target_names=CLASS_NAMES,output_dict=True,zero_division=0); pd.DataFrame(report).T.reset_index().rename(columns={'index':'class_or_average'}).to_csv(SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_Classification_Report.csv',index=False)
ensemble_pc,_=save_evaluation_artifacts(SUMMARY_DIR,'ensemble',y_true,metrics['predictions'],probabilities,ensemble)
shutil.copy2(SUMMARY_DIR/'confusion_matrix_ensemble.png',SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_ConfusionMatrix.png'); shutil.copy2(SUMMARY_DIR/'confusion_matrix_ensemble_normalized.png',SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_ConfusionMatrix_Normalized.png'); shutil.copy2(SUMMARY_DIR/'roc_ensemble.png',SUMMARY_DIR/'EfficientNet_B0_5Model_Ensemble_ROC.png')

hybrid_rows=[]
hybrid_values={'Validation':{'Accuracy':.9043,'Macro Precision':.9041,'Macro Recall':.8937,'Macro F1':.8979,'MCC':np.nan,'Macro AUC':.9723,'SMC F1':.9048},'Test':{'Accuracy':.9179,'Macro Precision':.9299,'Macro Recall':.8891,'Macro F1':.9056,'MCC':np.nan,'Macro AUC':.9507,'SMC F1':.8511}}
for context in ['Validation','Test']:
    ef_metrics=validation_records[1] if context=='Validation' else test_records[1]; ef_pc=validation_classwise[1] if context=='Validation' else test_classwise[1]
    hybrid_rows.append({'Context':f'Fold 1 {context}','Model':'Current hybrid (verified supplied result)',**hybrid_values[context]})
    hybrid_rows.append({'Context':f'Fold 1 {context}','Model':'3D EfficientNet-B0',**{m:float(ef_metrics.get(m,ef_metrics.get(m.lower().replace(' ','_')))) for m in METRICS},'SMC F1':float(ef_pc.loc[ef_pc['class']=='SMC','f1'].iloc[0])})
hybrid_test_path=PROJECT_ROOT/'Results'/'Proposed_Model'/'test_metrics_all_folds.csv'; hybrid_test=pd.read_csv(hybrid_test_path)
assert len(hybrid_test)==5 and np.isclose(hybrid_test.loc[hybrid_test.fold==1,'acc'].iloc[0],hybrid_values['Test']['Accuracy'],atol=5e-5) and np.isclose(hybrid_test.loc[hybrid_test.fold==1,'f1'].iloc[0],hybrid_values['Test']['Macro F1'],atol=5e-5)
hybrid_map={'Accuracy':'acc','Macro Precision':'prec','Macro Recall':'rec','Macro F1':'f1','Macro AUC':'auc'}
hybrid_rows.extend([{'Context':'Fixed test five checkpoints','Model':'Current hybrid exact mean','Source':str(hybrid_test_path),**{m:float(hybrid_test[c].mean()) for m,c in hybrid_map.items()},'MCC':np.nan,'SMC F1':np.nan},{'Context':'Fixed test five checkpoints','Model':'Current hybrid exact sample SD','Source':str(hybrid_test_path),**{m:float(hybrid_test[c].std(ddof=1)) for m,c in hybrid_map.items()},'MCC':np.nan,'SMC F1':np.nan},{'Context':'Fixed test five checkpoints','Model':'3D EfficientNet-B0 mean','Source':str(SUMMARY_DIR/'EfficientNet_B0_5Fold_Test_Summary.csv'),**{m:float(test_fold_frame[m].mean()) for m in METRICS},'SMC F1':np.nan},{'Context':'Fixed test five checkpoints','Model':'3D EfficientNet-B0 sample SD','Source':str(SUMMARY_DIR/'EfficientNet_B0_5Fold_Test_Summary.csv'),**{m:float(test_fold_frame[m].std(ddof=1)) for m in METRICS},'SMC F1':np.nan}])
hybrid_comparison=pd.DataFrame(hybrid_rows); hybrid_comparison.to_csv(SUMMARY_DIR/'EfficientNet_vs_Current_Hybrid_Comparison.csv',index=False)
smc_test=[]
for fold in FOLDS:
    r=test_classwise[fold].set_index('class').loc['SMC']; smc_test.append({'Fold':f'Fold {fold}','Support':float(r.support),'Precision':float(r.precision),'Recall':float(r.recall),'F1':float(r.f1),'AUC':float(r.auc)})
smc_test_frame=pd.DataFrame(smc_test); smc_test_summary=pd.concat([smc_test_frame,pd.DataFrame([{'Fold':'Mean',**{c:smc_test_frame[c].mean() for c in ['Support','Precision','Recall','F1','AUC']}},{'Fold':'SD',**{c:smc_test_frame[c].std(ddof=1) for c in ['Support','Precision','Recall','F1','AUC']}}])],ignore_index=True); smc_test_summary.to_csv(SUMMARY_DIR/'EfficientNet_B0_SMC_Test_Summary.csv',index=False)
hybrid_smc=[]; efficient_test_truth_order=np.array([LABEL_MAP[x] for x in pd.read_csv(RESULTS_ROOT/'fold_1'/'test_predictions.csv').true_label])
for fold in FOLDS:
    hp=pd.read_csv(PROJECT_ROOT/'Results'/'Proposed_Model'/f'fold_{fold}'/f'probs_fold_{fold}.csv'); assert len(hp)==719 and np.array_equal(hp.true.to_numpy(),efficient_test_truth_order)
    hpred=hp[[f'prob_{name}' for name in CLASS_NAMES]].to_numpy().argmax(axis=1); hp_,hr_,hf_,hs_=precision_recall_fscore_support(hp.true,hpred,labels=np.arange(NUM_CLASSES),zero_division=0); hybrid_smc.append({'Fold':fold,'Precision':hp_[5],'Recall':hr_[5],'F1':hf_[5],'Support':hs_[5]})
hybrid_smc_frame=pd.DataFrame(hybrid_smc); assert np.isclose(hybrid_smc_frame.loc[hybrid_smc_frame.Fold==1,'F1'].iloc[0],.8511,atol=5e-5)
smc_comparison=pd.DataFrame([{'Dataset':'Validation','EfficientNet estimate':'five-fold mean ± SD','EfficientNet F1':smc_frame.F1.mean(),'EfficientNet F1 SD':smc_frame.F1.std(ddof=1),'EfficientNet Recall':smc_frame.Recall.mean(),'EfficientNet Recall SD':smc_frame.Recall.std(ddof=1),'Hybrid exact available result':'Fold-1 only','Hybrid F1':.9048,'Hybrid F1 SD':np.nan,'Hybrid Recall':np.nan,'Hybrid Recall SD':np.nan},{'Dataset':'Fixed test','EfficientNet estimate':'five checkpoint mean ± SD on same cohort','EfficientNet F1':smc_test_frame.F1.mean(),'EfficientNet F1 SD':smc_test_frame.F1.std(ddof=1),'EfficientNet Recall':smc_test_frame.Recall.mean(),'EfficientNet Recall SD':smc_test_frame.Recall.std(ddof=1),'Hybrid exact available result':'five saved checkpoint probability files on same 719-label order','Hybrid F1':hybrid_smc_frame.F1.mean(),'Hybrid F1 SD':hybrid_smc_frame.F1.std(ddof=1),'Hybrid Recall':hybrid_smc_frame.Recall.mean(),'Hybrid Recall SD':hybrid_smc_frame.Recall.std(ddof=1)}]); smc_comparison.to_csv(SUMMARY_DIR/'EfficientNet_B0_SMC_vs_Current_Hybrid.csv',index=False)

runtime_rows=[]
for fold in FOLDS:
    r=json.loads((RESULTS_ROOT/f'fold_{fold}'/'runtime_summary.json').read_text()); runtime_rows.append({'Fold':fold,'Parameters':PARAMETERS,'Checkpoint size MiB':(RESULTS_ROOT/f'fold_{fold}'/'best_model.pt').stat().st_size/2**20,'Peak GPU memory GiB':r['peak_gpu_memory_gib'],'Average epoch minutes':r['average_epoch_minutes'],'Total training minutes':r['total_training_minutes'],'Inference seconds per MRI':test_runtime[fold-1]['inference_seconds_per_mri']})
complexity=pd.DataFrame(runtime_rows); complexity.loc[len(complexity)]={'Fold':'Mean','Parameters':PARAMETERS,'Checkpoint size MiB':complexity['Checkpoint size MiB'].mean(),'Peak GPU memory GiB':complexity['Peak GPU memory GiB'].mean(),'Average epoch minutes':complexity['Average epoch minutes'].mean(),'Total training minutes':complexity['Total training minutes'].sum(),'Inference seconds per MRI':complexity['Inference seconds per MRI'].mean()}; complexity.to_csv(SUMMARY_DIR/'EfficientNet_B0_5Fold_Complexity.csv',index=False)
vmean=validation_summary[validation_summary.Fold=='Mean'].iloc[0]; vsd=validation_summary[validation_summary.Fold=='SD'].iloc[0]; tmean=test_summary[test_summary.Fold=='Mean'].iloc[0]; tsd=test_summary[test_summary.Fold=='SD'].iloc[0]; pc=test_classwise_summary.set_index('Class')
manuscript={'Model':'3D EfficientNet-B0'}
for m in METRICS: manuscript[f'Validation {m} mean±SD']=f'{vmean[m]:.6f} ± {vsd[m]:.6f}'; manuscript[f'Test {m} mean±SD']=f'{tmean[m]:.6f} ± {tsd[m]:.6f}'
for name in CLASS_NAMES: manuscript[f'{name} F1']=float(pc.loc[name,'Mean F1'])
manuscript.update({'Parameters':PARAMETERS,'Peak GPU memory':float(complexity.iloc[:5]['Peak GPU memory GiB'].max()),'Runtime':float(complexity.iloc[:5]['Total training minutes'].sum())}); pd.DataFrame([manuscript]).to_csv(SUMMARY_DIR/'EfficientNet_B0_Final_Manuscript_Summary.csv',index=False)
gpu_audit['completed_utc']=datetime.now(timezone.utc).isoformat(); gpu_audit['valid']=True; gpu_audit['all_folds_strictly_sequential']=True; gpu_audit['checkpoints_frozen_before_test']=True; gpu_audit['final_gpu']=gpu_snapshot('complete'); json_dump(SUMMARY_DIR/'EfficientNet_B0_GPU_Sequential_Audit.json',gpu_audit)
json_dump(SUMMARY_DIR/'EfficientNet_B0_5Fold_Integrity.json',{'valid':True,'one_notebook_only':True,'fold1_reused':True,'folds_2_to_5_trained_from_scratch':True,'configuration_frozen_before_training':True,'checkpoints_frozen_before_test':True,'same_fixed_test_719_for_all_checkpoints':True,'combined_test_prediction_rows':len(combined_test),'no_tuning':True,'no_additional_models':True,'manuscript_modified':False})
print('\nSECONDARY FIVE-MODEL ENSEMBLE')
print(ensemble_metrics.to_string(index=False))
print('\nEFFICIENTNET VS CURRENT HYBRID (exact Fold-1 contexts only)')
print(hybrid_comparison.to_string(index=False))
print('\nFINAL MANUSCRIPT SUMMARY')
print(pd.DataFrame([manuscript]).to_string(index=False))
print('\nWORKFLOW COMPLETE — no manuscript edits; no additional models.')


SECONDARY FIVE-MODEL ENSEMBLE
                           Fold  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC                    Analysis
Five-model probability ensemble  0.963839         0.963892      0.925457  0.942045 0.953925   0.991373 Secondary ensemble analysis

EFFICIENTNET VS CURRENT HYBRID (exact Fold-1 contexts only)
                    Context                                     Model  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC   SMC F1                                                                                                                                                                         Source
          Fold 1 Validation Current hybrid (verified supplied result)  0.904300         0.904100      0.893700  0.897900      NaN   0.972300 0.904800                                                                                                                                                                            

In [2]:
import copy, io, json, math, time, platform
from datetime import datetime, timezone
from pathlib import Path

PROFILE_PROJECT_ROOT = Path("/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM")
PROFILE_EXPECTED_PARAMETERS = 4_696_066

def run_efficientnet_b0_complexity_profile():
    try:
        import torch
        import torch.nn as nn
        import numpy as np
        import pandas as pd
        import monai
        from monai.networks.nets import EfficientNetBN
    except Exception as exc:
        print("Profiling dependencies unavailable:", exc)
        return

    ns = globals()
    candidate_names = ["best_model", "model", "net", "classifier", "efficientnet_model"]
    candidates = [(name, ns[name]) for name in candidate_names if name in ns and isinstance(ns[name], nn.Module)]
    valid_candidates = [(name, obj) for name, obj in candidates if sum(p.numel() for p in obj.parameters()) == PROFILE_EXPECTED_PARAMETERS]

    if torch.cuda.is_available():
        profile_gpu = 2 if torch.cuda.device_count() > 2 else (1 if torch.cuda.device_count() > 1 else 0)
        device = torch.device(f"cuda:{profile_gpu}")
    else:
        device = torch.device("cpu")

    if valid_candidates:
        model_name, model_object = next(((n, o) for n, o in valid_candidates if n == "best_model"), valid_candidates[0])
        profile_model = copy.deepcopy(model_object).to(device)
        architecture_source = f"Valid live EfficientNet-B0 model object: {model_name}"
        print("Valid live EfficientNet-B0 object found:", model_name)
    else:
        profile_model = EfficientNetBN(
            "efficientnet-b0",
            pretrained=False,
            spatial_dims=3,
            in_channels=1,
            num_classes=6,
        ).to(device)
        architecture_source = "Exact finalized MONAI EfficientNetBN configuration copied from the experiment notebook"
        print("No valid live EfficientNet-B0 object was available.")
        print("Instantiated the exact finalized architecture locally for profiling only.")

    dummy = train_x = train_y = None
    total = None
    integrity = "NOT RUN"
    try:
        profile_model.eval()
        cuda = device.type == "cuda" and torch.cuda.is_available()
        total = int(sum(p.numel() for p in profile_model.parameters()))
        print("Architecture source:", architecture_source)
        print("Profiling device:", device)
        print("EXPECTED PARAMETERS: 4,696,066")
        print(f"OBSERVED PARAMETERS: {total:,}")
        if total != PROFILE_EXPECTED_PARAMETERS:
            integrity = "FAIL"
            print("ARCHITECTURE INTEGRITY CHECK FAILED.")
            print("Profiling results were NOT generated.")
            print("ORIGINAL CELLS MODIFIED: NO")
            print("ORIGINAL CELLS EXECUTED: NO")
            print("ORIGINAL OUTPUTS CLEARED: NO")
            print("TRAINING EXECUTED: NO")
            print("VALIDATION EXECUTED: NO")
            print("DATASET TEST INFERENCE EXECUTED: NO")
            print("CHECKPOINT LOADED: NO")
            return
        integrity = "PASS"
        print("ARCHITECTURE INTEGRITY CHECK: PASSED")

        dummy = torch.zeros((1, 1, 128, 128, 128), dtype=torch.float32, device=device)
        def logits(value):
            if isinstance(value, torch.Tensor):
                return value
            if isinstance(value, dict):
                for key in ("logits", "output", "out"):
                    if isinstance(value.get(key), torch.Tensor):
                        return value[key]
                return next((x for x in value.values() if isinstance(x, torch.Tensor)), None)
            if isinstance(value, (tuple, list)):
                return next((x for x in value if isinstance(x, torch.Tensor)), None)
            return None

        with torch.inference_mode():
            sanity = logits(profile_model(dummy))
        if sanity is None or tuple(sanity.shape) != (1, 6):
            print("SYNTHETIC FORWARD CHECK FAILED; expected output shape (1, 6).")
            print("Observed output:", None if sanity is None else tuple(sanity.shape))
            print("Profiling results were NOT generated.")
            return
        print("SYNTHETIC FORWARD CHECK: PASSED")
        print("Synthetic output shape:", tuple(sanity.shape))

        trainable = int(sum(p.numel() for p in profile_model.parameters() if p.requires_grad))
        param_mib = total * 4 / 1024**2
        state_buffer = io.BytesIO()
        torch.save(profile_model.state_dict(), state_buffer)
        state_mib = state_buffer.tell() / 1024**2

        operation_count = {"macs": 0.0}
        hooks = []
        def conv_hook(layer, inputs, output):
            output = logits(output)
            if output is not None:
                operation_count["macs"] += output.numel() * math.prod(layer.kernel_size) * layer.in_channels / layer.groups
        def linear_hook(layer, inputs, output):
            output = logits(output)
            if output is not None:
                operation_count["macs"] += output.numel() * layer.in_features
        for layer in profile_model.modules():
            if isinstance(layer, nn.Conv3d):
                hooks.append(layer.register_forward_hook(conv_hook))
            elif isinstance(layer, nn.Linear):
                hooks.append(layer.register_forward_hook(linear_hook))
        try:
            with torch.inference_mode():
                logits(profile_model(dummy))
        finally:
            for hook in hooks:
                hook.remove()
        macs = float(operation_count["macs"])
        flops = 2.0 * macs

        inference_allocated = inference_reserved = None
        latencies = []
        if cuda:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            torch.cuda.synchronize(device)
            with torch.inference_mode():
                logits(profile_model(dummy))
            torch.cuda.synchronize(device)
            inference_allocated = torch.cuda.max_memory_allocated(device) / 1024**2
            inference_reserved = torch.cuda.max_memory_reserved(device) / 1024**2
            for _ in range(10):
                with torch.inference_mode():
                    logits(profile_model(dummy))
            torch.cuda.synchronize(device)
            for _ in range(50):
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
                with torch.inference_mode():
                    logits(profile_model(dummy))
                end.record()
                end.synchronize()
                latencies.append(float(start.elapsed_time(end)))
        else:
            print("CUDA is unavailable; using CPU wall-clock timing.")
            for _ in range(10):
                with torch.inference_mode():
                    logits(profile_model(dummy))
            for _ in range(50):
                start_time = time.perf_counter()
                with torch.inference_mode():
                    logits(profile_model(dummy))
                latencies.append((time.perf_counter() - start_time) * 1000)

        latencies = np.asarray(latencies, dtype=float)
        mean_ms = float(latencies.mean())
        sd_ms = float(latencies.std(ddof=1))
        median_ms = float(np.median(latencies))
        p95_ms = float(np.percentile(latencies, 95))
        min_ms = float(latencies.min())
        max_ms = float(latencies.max())
        throughput = 1000 / mean_ms if mean_ms > 0 else None

        backward_allocated = backward_reserved = None
        if cuda:
            try:
                profile_model.train()
                train_x = torch.zeros((1, 1, 128, 128, 128), dtype=torch.float32, device=device)
                train_y = torch.zeros((1,), dtype=torch.long, device=device)
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(device)
                profile_model.zero_grad(set_to_none=True)
                loss = nn.CrossEntropyLoss()(logits(profile_model(train_x)), train_y)
                loss.backward()
                torch.cuda.synchronize(device)
                backward_allocated = torch.cuda.max_memory_allocated(device) / 1024**2
                backward_reserved = torch.cuda.max_memory_reserved(device) / 1024**2
            except Exception as exc:
                print("Synthetic forward+backward peak memory: NOT AVAILABLE", exc)
                torch.cuda.empty_cache()
            finally:
                profile_model.eval()

        active_results = PROFILE_PROJECT_ROOT / "Results" / "Baselines" / "3DEfficientNet_ImageID_5Fold"
        history_files = sorted(active_results.glob("fold_*/history.csv"))
        runtime_files = sorted(active_results.glob("fold_*/runtime_summary.json"))
        epoch_minutes = []
        for history_path in history_files:
            history = pd.read_csv(history_path)
            if "epoch_minutes" in history.columns:
                epoch_minutes.extend(pd.to_numeric(history["epoch_minutes"], errors="coerce").dropna().tolist())
        logged_fold_minutes = []
        for runtime_path in runtime_files:
            runtime_record = json.loads(runtime_path.read_text())
            if runtime_record.get("total_training_minutes") is not None:
                logged_fold_minutes.append(float(runtime_record["total_training_minutes"]))
        if epoch_minutes and logged_fold_minutes:
            epoch_array = np.asarray(epoch_minutes, dtype=float)
            historical_epoch_mean_min = float(epoch_array.mean())
            historical_epoch_sd_min = float(epoch_array.std(ddof=1)) if len(epoch_array) > 1 else 0.0
            historical_total_min = float(sum(logged_fold_minutes))
            historical_epochs = int(len(epoch_array))
            training_time_note = "Exact historical timing read from active history.csv and runtime_summary.json files; no retraining or timestamp estimation."
        else:
            historical_epoch_mean_min = historical_epoch_sd_min = historical_total_min = None
            historical_epochs = None
            training_time_note = "Historical training time was not available in active saved logs and was not estimated."

        if cuda:
            gpu = torch.cuda.get_device_name(device)
            cuda_version = torch.version.cuda
            gpu_index = device.index
            total_gpu_mib = torch.cuda.get_device_properties(device).total_memory / 1024**2
        else:
            gpu, cuda_version, gpu_index, total_gpu_mib = "CUDA unavailable", torch.version.cuda, None, None

        output_dir = PROFILE_PROJECT_ROOT / "Results" / "Analysis" / "Complexity_Runtime" / "EfficientNetB0"
        output_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        def unused_path(filename):
            base = output_dir / filename
            if not base.exists():
                return base
            candidate = output_dir / f"{base.stem}_{stamp}{base.suffix}"
            counter = 1
            while candidate.exists():
                candidate = output_dir / f"{base.stem}_{stamp}_{counter}{base.suffix}"
                counter += 1
            return candidate
        output_paths = {
            "summary_csv": unused_path("complexity_runtime_summary.csv"),
            "summary_json": unused_path("complexity_runtime_summary.json"),
            "latency_csv": unused_path("inference_latency_raw_ms.csv"),
            "environment_json": unused_path("complexity_runtime_environment.json"),
        }

        summary = {
            "Profiling_Label": "Architecture-level FP32 profiling of the finalized baseline 3D EfficientNet-B0",
            "Model": "3D EfficientNet-B0",
            "Model_Class": type(profile_model).__name__,
            "Architecture": "MONAI EfficientNetBN efficientnet-b0, pretrained=False, spatial_dims=3, in_channels=1, num_classes=6",
            "Architecture_Source": architecture_source,
            "Expected_Parameters": PROFILE_EXPECTED_PARAMETERS,
            "Architecture_Integrity_Check": integrity,
            "Trained_Weights_Loaded": False,
            "Checkpoint_Loaded": False,
            "Input_Shape": "(1, 1, 128, 128, 128)",
            "Batch_Size": 1,
            "Precision": "FP32",
            "Warmup_Runs": 10,
            "Timed_Runs": 50,
            "Total_Parameters": total,
            "Trainable_Parameters": trainable,
            "Parameter_Memory_MiB": param_mib,
            "Serialized_StateDict_MiB": state_mib,
            "MACs": macs,
            "Approx_FLOPs": flops,
            "Inference_Mean_ms_per_MRI": mean_ms,
            "Inference_SD_ms": sd_ms,
            "Inference_Median_ms": median_ms,
            "Inference_P95_ms": p95_ms,
            "Inference_Min_ms": min_ms,
            "Inference_Max_ms": max_ms,
            "Inference_Throughput_MRI_per_s": throughput,
            "Inference_Peak_Allocated_MiB": inference_allocated,
            "Inference_Peak_Reserved_MiB": inference_reserved,
            "Synthetic_Forward_Backward_Peak_Allocated_MiB": backward_allocated,
            "Synthetic_Forward_Backward_Peak_Reserved_MiB": backward_reserved,
            "Historical_Epochs_Logged": historical_epochs,
            "Historical_Mean_Epoch_Time_min": historical_epoch_mean_min,
            "Historical_SD_Epoch_Time_min": historical_epoch_sd_min,
            "Historical_Total_Training_Time_min": historical_total_min,
            "Historical_Training_Time_Note": training_time_note,
            "Historical_History_Files": [str(p) for p in history_files],
            "Historical_Runtime_Files": [str(p) for p in runtime_files],
            "GPU": gpu,
            "CUDA_Device_Index": gpu_index,
            "Total_GPU_Memory_MiB": total_gpu_mib,
            "PyTorch": torch.__version__,
            "MONAI": monai.__version__,
            "CUDA": cuda_version,
            "MAC_Convention": "Conv3d and Linear only; one MAC is approximately two FLOPs.",
            "Runtime_Excludes": "MRI preprocessing and disk I/O",
        }
        pd.DataFrame([summary]).to_csv(output_paths["summary_csv"], index=False)
        output_paths["summary_json"].write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
        pd.DataFrame({"latency_ms": latencies}).to_csv(output_paths["latency_csv"], index=False)
        environment = {
            "Profiling_Label": summary["Profiling_Label"],
            "GPU": gpu, "CUDA_Device_Index": gpu_index, "CUDA": cuda_version,
            "Total_GPU_Memory_MiB": total_gpu_mib, "PyTorch": torch.__version__,
            "MONAI": monai.__version__, "Python": platform.python_version(),
            "Input_Shape": [1, 1, 128, 128, 128], "Batch_Size": 1, "Precision": "FP32",
            "Warmup_Runs": 10, "Timed_Runs": 50,
            "Runtime_Excludes": "MRI preprocessing and disk I/O",
            "Trained_Weights_Loaded": False, "Checkpoint_Loaded": False,
        }
        output_paths["environment_json"].write_text(json.dumps(environment, indent=2, default=str), encoding="utf-8")

        print("ORIGINAL CELLS MODIFIED: NO")
        print("ORIGINAL CELLS EXECUTED: NO")
        print("ORIGINAL OUTPUTS CLEARED: NO")
        print("TRAINING EXECUTED: NO")
        print("VALIDATION EXECUTED: NO")
        print("DATASET TEST INFERENCE EXECUTED: NO")
        print("CHECKPOINT LOADED: NO")
        print("\nARCHITECTURE SOURCE:")
        print("Exact finalized MONAI EfficientNetBN configuration copied from the experiment notebook")
        print("\nEXPECTED PARAMETERS:")
        print("4,696,066")
        print("\nOBSERVED PARAMETERS:")
        print(f"{total:,}")
        print("\nARCHITECTURE INTEGRITY CHECK:")
        print("PASS")
        print("\n==================================================")
        print("Architecture-level FP32 profiling of the finalized baseline 3D EfficientNet-B0")
        print("==================================================")
        print("Model:", summary["Model"])
        print("Model class:", summary["Model_Class"])
        print("Input:", summary["Input_Shape"])
        print("Precision:", summary["Precision"])
        print("GPU:", summary["GPU"])
        print("\nTotal parameters:", total)
        print("Trainable parameters:", trainable)
        print("Serialized state_dict size:", f"{state_mib:.3f} MiB")
        print("Parameter memory:", f"{param_mib:.3f} MiB")
        print("\nMACs / MRI:", f"{macs:.0f}")
        print("Approximate FLOPs / MRI:", f"{flops:.0f}", "(one MAC is approximately two FLOPs; Conv3d/Linear only)")
        print("\nFP32 inference, batch size 1:")
        print("Mean latency:", f"{mean_ms:.4f} ms/MRI")
        print("SD:", f"{sd_ms:.4f} ms")
        print("Median:", f"{median_ms:.4f} ms")
        print("P95:", f"{p95_ms:.4f} ms")
        print("Throughput:", f"{throughput:.4f} MRI/s")
        print("\nInference peak allocated GPU memory:", "NOT AVAILABLE" if inference_allocated is None else f"{inference_allocated:.3f} MiB")
        print("Inference peak reserved GPU memory:", "NOT AVAILABLE" if inference_reserved is None else f"{inference_reserved:.3f} MiB")
        print("\nSynthetic forward+backward peak allocated memory:", "NOT AVAILABLE" if backward_allocated is None else f"{backward_allocated:.3f} MiB")
        print("Synthetic forward+backward peak reserved memory:", "NOT AVAILABLE" if backward_reserved is None else f"{backward_reserved:.3f} MiB")
        print("\nHistorical saved training-time information:")
        print("Logged epochs:", "NOT AVAILABLE" if historical_epochs is None else historical_epochs)
        print("Mean epoch time:", "NOT AVAILABLE" if historical_epoch_mean_min is None else f"{historical_epoch_mean_min:.6f} min")
        print("Epoch-time SD:", "NOT AVAILABLE" if historical_epoch_sd_min is None else f"{historical_epoch_sd_min:.6f} min")
        print("Five-fold total training time:", "NOT AVAILABLE" if historical_total_min is None else f"{historical_total_min:.6f} min")
        print(training_time_note)
        print("\nIMPORTANT INTERPRETATION:")
        print("- Runtime excludes MRI preprocessing and disk I/O.")
        print("- Inference uses batch size 1 and FP32.")
        print("- FLOPs use the same Conv3d/Linear convention as the Proposed-model profile.")
        print("- No training, validation, or dataset-level inference was run.")
        print("- No checkpoint or trained weights were loaded; learned values are not required for graph-level profiling.")
        print("\nOutput directory:", output_dir.resolve())
        print("Output files:")
        for output_path in output_paths.values():
            print(output_path.resolve())
        print("==================================================")
    except Exception as exc:
        print("Profiling cell stopped gracefully:", type(exc).__name__, exc)
    finally:
        if profile_model is not None:
            profile_model.zero_grad(set_to_none=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

run_efficientnet_b0_complexity_profile()


No valid live EfficientNet-B0 object was available.
Instantiated the exact finalized architecture locally for profiling only.
Architecture source: Exact finalized MONAI EfficientNetBN configuration copied from the experiment notebook
Profiling device: cuda:2
EXPECTED PARAMETERS: 4,696,066
OBSERVED PARAMETERS: 4,696,066
ARCHITECTURE INTEGRITY CHECK: PASSED
SYNTHETIC FORWARD CHECK: PASSED
Synthetic output shape: (1, 6)
ORIGINAL CELLS MODIFIED: NO
ORIGINAL CELLS EXECUTED: NO
ORIGINAL OUTPUTS CLEARED: NO
TRAINING EXECUTED: NO
VALIDATION EXECUTED: NO
DATASET TEST INFERENCE EXECUTED: NO
CHECKPOINT LOADED: NO

ARCHITECTURE SOURCE:
Exact finalized MONAI EfficientNetBN configuration copied from the experiment notebook

EXPECTED PARAMETERS:
4,696,066

OBSERVED PARAMETERS:
4,696,066

ARCHITECTURE INTEGRITY CHECK:
PASS

Architecture-level FP32 profiling of the finalized baseline 3D EfficientNet-B0
Model: 3D EfficientNet-B0
Model class: EfficientNetBN
Input: (1, 1, 128, 128, 128)
Precision: FP32
GP